# Identity → comic panels, + a stronger three-way (flux-recipes)
Two compositions on the residual-identity seam:
1. **`identity_story`** — a FACE from a reference photo across comic scenes (PuLID identity ⊕ story K/V share): one
   character, many panels, face-locked.
2. **`identity_structure_appearance`** with a proper **texture** appearance donor (a stone/brick material, not an
   object) — so the material actually comes through while identity + structure hold.
OPEN-WEIGHT (PuLID). Runtime: 80GB A100.

In [ ]:
import subprocess, os
for _ in range(3):
    if subprocess.call(["pip","install","-q","git+https://github.com/huggingface/diffusers.git"])==0: break
!pip install -q transformers accelerate sentencepiece protobuf hf_transfer scikit-image
!pip install -q insightface facexlib onnxruntime-gpu timm einops ftfy opencv-python-headless
subprocess.run(["rm","-rf","flux-recipes"])
subprocess.run(["git","clone","-q","-b","main","https://github.com/remyxai/flux-recipes.git"])

In [ ]:
import sys, torch, numpy as np, cv2
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
sys.path.insert(0,"flux-recipes")
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
assert torch.cuda.is_available(); print("GPU:",torch.cuda.get_device_name(0))
from flux_modular import RecipeRunner
runner=RecipeRunner(steps=20)
from PIL import Image, ImageDraw
from skimage import data
from IPython.display import display
ID=Image.fromarray(data.astronaut()).convert("RGB").resize((1024,1024))   # the character's face
print("runner ready")

## 1. Identity → comic panels (`identity_story`) — one face, many scenes

In [ ]:
scenes=["a close portrait, reading a book in a cozy cafe #Cafe","a portrait resting on a forest trail at dawn #Forest","a portrait at the controls of a spaceship cockpit #Space"]
panels=runner.run("identity_story", {"id_image":ID,"scene_prompts":scenes}, id_weight=1.0, seed=0)
# ArcFace-sim of each panel to the reference face (the encoder was loaded by the run)
from flux_modular.identity import _PULID
enc=_PULID["enc"]
def arc_emb(pil):
    fi=enc.app.get(cv2.cvtColor(np.asarray(pil.convert("RGB")),cv2.COLOR_RGB2BGR))
    if not fi: return None
    fi=sorted(fi,key=lambda x:(x['bbox'][2]-x['bbox'][0])*(x['bbox'][3]-x['bbox'][1]))[-1]
    e=fi['embedding']; return e/(np.linalg.norm(e)+1e-9)
_ref=arc_emb(ID)
def arc(im):
    e=arc_emb(im); return float(np.dot(e,_ref)) if e is not None else None
def fmt(x): return f"{x:.2f}" if x is not None else "no-face"
caps=[s.partition("#")[2].strip() for s in scenes]
# comic strip: reference chip + panels with captions + face-sim
tiles=[("reference",ID,"")]+[(caps[i],panels[i],f"face={fmt(arc(panels[i]))}") for i in range(len(panels))]
c=340; strip=Image.new("RGB",(len(tiles)*c+(len(tiles)+1)*8, c+40),"white"); d=ImageDraw.Draw(strip)
for j,(cap,im,s) in enumerate(tiles):
    x=8+j*(c+8); strip.paste(im.resize((c,c)),(x,4)); d.text((x+4,c+8),cap[:34],fill="black"); d.text((x+4,c+22),s,fill="black")
display(strip)
print("face-lock across panels: the same character (from one photo) in each scene, ArcFace-sim high per panel.")

## 2. Stronger three-way — a TEXTURE appearance donor (material, not an object)

In [ ]:
from transformers import CLIPModel, CLIPProcessor, pipeline as hf_pipeline
TEX=Image.fromarray(data.brick()).convert("RGB").resize((1024,1024))   # stone/brick material (a texture, not an object)
PROMPT="a portrait of a person, studio lighting"
_dep=hf_pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0)
def dcorr(a,b):
    da=np.asarray(_dep(a)["depth"].convert("L").resize((256,256)),np.float32).ravel(); db=np.asarray(_dep(b)["depth"].convert("L").resize((256,256)),np.float32).ravel()
    return float(np.corrcoef(da,db)[0,1])
_clip=CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda").eval(); _cp=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
@torch.no_grad()
def clip_img(a,b):
    px=_cp(images=[a,b],return_tensors="pt").to("cuda"); v=_clip.vision_model(pixel_values=px["pixel_values"]).pooler_output
    e=_clip.visual_projection(v); e=e/e.norm(dim=-1,keepdim=True); return float((e[0]@e[1]).cpu())
panels=[("id + structure",ID,None),("texture (material)",TEX,None)]
for rx in (0.0, 0.3, 0.5):   # HONEST: Redux swamps at strength for ANY donor -> appearance is a subtle accent
    im=runner.run("identity_structure_appearance", {"id_image":ID,"ref_structure":ID,"ref_appearance":TEX,"prompt":PROMPT},
                  id_weight=1.0, S=0.5, redux_scale=rx, seed=0)
    panels.append((f"redux={rx}", im, f"face={fmt(arc(im))} d={dcorr(im,ID):.2f} tex={clip_img(im,TEX):.2f}"))
c=250; g=Image.new("RGB",(len(panels)*c+(len(panels)+1)*6,c+40),"white"); dr=ImageDraw.Draw(g)
for j,(n,im,s) in enumerate(panels):
    x=6+j*(c+6); g.paste(im.resize((c,c)),(x,4)); dr.text((x+4,c+8),str(n)[:30],fill="black")
    if s: dr.text((x+4,c+22),str(s)[:30],fill="black")
display(g)
print("FINDING: even a texture swamps at redux>=0.4 (a brick WALL takes over, no face) — Redux injects the")
print("reference GLOBAL content regardless of donor. So the three-way appearance is a SUBTLE accent only (redux~0.3):")
print("identity+structure hold, palette nudges. Strong material transfer onto a face needs a non-Redux channel.")